## Reading flight_bookings from the delta lake

# Applying The transformations

In [0]:
from pyspark.sql.functions import (
    col, trim, upper, to_date, to_timestamp,
    when, date_format, to_timestamp, concat, lit,
    coalesce, expr  # Added missing imports
)
from pyspark.sql import functions as F

# =============================================================
# 1) READ BRONZE DELTA TABLE
# =============================================================
flight_bookings_bronze_path = "s3://travel-analytics-bronze/delta/bronze/flight_bookings/"
bronze_df = spark.read.format("delta").load(flight_bookings_bronze_path)

# =============================================================
# 2) SILVER TRANSFORMATIONS (CHAINED SEQUENCE)
# =============================================================
flight_bookings_silver_df = (
    bronze_df
    
    # ---------------------------------------------------------
    # A) Cast to Correct Data Types
    # ---------------------------------------------------------
    .withColumn("trip_id", col("trip_id").cast("int"))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("airline_id", col("airline_id").cast("int"))
    .withColumn("aircraft_id", col("aircraft_id").cast("string"))
    .withColumn("airport_src", col("airport_src").cast("int"))
    .withColumn("airport_dst", col("airport_dst").cast("int"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("discount_amount", col("discount_amount").cast("double"))
    
    # ---------------------------------------------------------
    # B) Handle Time Fields
    # ---------------------------------------------------------
    .withColumn("departure_time", col("departure_time").cast("string"))
    .withColumn("arrival_time", col("arrival_time").cast("string"))
    .withColumn("booking_time", col("booking_time").cast("string"))
    
    # Fix flight_duration padding
    .withColumn("flight_duration", 
        when(col("flight_duration").rlike("^[0-9]:[0-9]{2}$"),
             concat(lit("0"), col("flight_duration")))
        .otherwise(col("flight_duration").cast("string"))
    )
    # ---------------------------------------------------------
    # B) Handle Time Fields
    # ---------------------------------------------------------
    .withColumn("departure_time", col("departure_time").cast("string"))
    .withColumn("arrival_time", col("arrival_time").cast("string"))
    .withColumn("booking_time", col("booking_time").cast("string"))    
    # ---------------------------------------------------------
    # D) Calculate Final Ticket Price & Clean Discounts
    # ---------------------------------------------------------
    .withColumn("discount_amount",
        when(col("discount_amount") < 0, None)
        .otherwise(col("discount_amount"))
    )
    .withColumn("final_ticket_price",  # Use lowercase for consistency
        (col("price") - coalesce(col("discount_amount"), lit(0))).cast("double")
    )
    
    # ---------------------------------------------------------
    # E) Clean & Standardize String Columns
    # ---------------------------------------------------------
    .withColumn("aircraft_id", trim(upper(col("aircraft_id"))))
    .withColumn("booking_status", trim(upper(col("booking_status"))))
    .withColumn("flight_number", trim(upper(col("flight_number"))))
    .withColumn("travel_class", trim(upper(col("travel_class"))))
    .withColumn("payment_method", trim(upper(col("payment_method"))))
    .withColumn("seat_number", trim(upper(col("seat_number"))))
    # ---------------------------------------
    # B) String Columns → UNKNOWN
    # ---------------------------------------
    .withColumn("payment_method", coalesce(col("payment_method"), lit("UNKNOWN")))
    .withColumn("booking_status", coalesce(col("booking_status"), lit("UNKNOWN")))
    .withColumn("travel_class",   coalesce(col("travel_class"), lit("UNKNOWN")))
    .withColumn("seat_number",    coalesce(col("seat_number"), lit("UNKNOWN")))
    # ---------------------------------------------------------
    # F) CDC Handling (optional but recommended)
    # ---------------------------------------------------------
    .withColumn("updated_at", to_timestamp(col("_ab_cdc_updated_at")))
    .withColumn("deleted_flag", col("_ab_cdc_deleted_at").isNotNull())
    
    # ---------------------------------------------------------
    # G) Remove Technical Columns
    # ---------------------------------------------------------
    .drop(
        "_airbyte_ab_id",
        "_airbyte_emitted_at",
        "_ab_cdc_lsn",
        "_airbyte_additional_properties"
    )
    
    # ---------------------------------------------------------
    # H) Deduplicate Using Composite PK
    # PK = (trip_id, flight_number, departure_date)
    # ---------------------------------------------------------
    .dropDuplicates(["trip_id", "flight_number", "departure_date"])
    .withColumn("deleted_flag", col("_ab_cdc_deleted_at").isNotNull())
)

# =============================================================
# 3) RENAME COLUMNS → Camel_Case (as required)
# =============================================================
rename_map = {
    "trip_id": "Trip_Id",
    "customer_id": "Customer_Id",
    "flight_number": "Flight_Number",
    "airline_id": "Airline_Id",
    "aircraft_id": "Aircraft_Id",
    "airport_src": "Airport_Src",
    "airport_dst": "Airport_Dst",
    "departure_time": "Departure_Time",
    "departure_date": "Departure_Date",
    "booking_time": "Booking_Time",
    "booking_date": "Booking_Date",
    "flight_duration": "Flight_Duration",
    "travel_class": "Travel_Class",
    "seat_number": "Seat_Number",
    "price": "Price",
    "arrival_date": "Arrival_Date",
    "arrival_time": "Arrival_Time",
    "payment_method": "Payment_Method",
    "booking_status": "Booking_Status",
    "discount_amount": "Discount_Amount",
    "final_ticket_price": "Final_Ticket_Price",  # Changed from "final_ticket_price"
    "updated_at": "Updated_At",
    "deleted_flag": "Deleted_Flag"
}

for old, new in rename_map.items():
    flight_bookings_silver_df = flight_bookings_silver_df.withColumnRenamed(old, new)

# =============================================================
# 4) FINAL COLUMN ORDER
# =============================================================
final_cols = [
    "Trip_Id", "Customer_Id", "Flight_Number", "Airline_Id", "Aircraft_Id",
    "Airport_Src", "Airport_Dst", "Departure_Time", "Departure_Date",
    "Booking_Time", "Booking_Date", "Flight_Duration", "Travel_Class",
    "Seat_Number", "Price", "Arrival_Date", "Arrival_Time",
    "Payment_Method", "Booking_Status", "Discount_Amount",
    "Final_Ticket_Price", "Updated_At", "Deleted_Flag"
]

# Select columns in correct order
flight_bookings_silver_df = flight_bookings_silver_df.select(*final_cols)

# =============================================================
# 5) WRITE TO SILVER LAYER
# =============================================================
silver_output_path = "s3://travel-analytics-bronze/delta/silver/flight_bookings/"

flight_bookings_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("Departure_Date") \
    .save(silver_output_path)

In [0]:
silver_path = "s3://travel-analytics-bronze/delta/silver/flight_bookings/"

df_silver = spark.read.format("delta").load(silver_path)

display(df_silver)

Trip_Id,Customer_Id,Flight_Number,Airline_Id,Aircraft_Id,Airport_Src,Airport_Dst,Departure_Time,Departure_Date,Booking_Time,Booking_Date,Flight_Duration,Travel_Class,Seat_Number,Price,Arrival_Date,Arrival_Time,Payment_Method,Booking_Status,Discount_Amount,Final_Ticket_Price,Updated_At,Deleted_Flag
174,1910557,W31330,20976,CR9CRJ9,34,75,13:30:00,2018-12-28,12:30:00,2018-12-27,01:17,BUSINESS,20C,457.0,2018-12-28,14:47:00,CREDIT CARD,COMPLETED,0.0,457.0,null,false
3759,7409190,FR0915,4296,738B738,15,40,09:15:00,2018-12-28,08:00:00,2018-12-27,02:38,ECONOMY,3D,155.0,2018-12-28,11:53:00,DEBIT CARD,CANCELLED,15.0,140.0,null,false
331,7538021,JJ0730,4867,320A320,21,15,07:30:00,2018-12-28,06:30:00,2018-12-27,01:09,ECONOMY,8E,47.0,2018-12-28,08:39:00,CREDIT CARD,COMPLETED,2.0,45.0,null,false
127,482158,BJ1900,3740,320A320,54,66,19:00:00,2018-12-28,18:00:00,2018-12-27,02:26,BUSINESS,33D,540.0,2018-12-28,21:26:00,CREDIT CARD,COMPLETED,0.0,540.0,null,false
3485,2811084,AZ1015,596,772B772,49,41,10:15:00,2018-12-28,10:00:00,2018-12-21,12:14,ECONOMY,6C,849.0,2018-12-28,22:29:00,DEBIT CARD,COMPLETED,80.0,769.0,null,false
3560,1796228,SK1100,4319,AT7AT72,41,49,11:00:00,2018-12-28,15:00:00,2018-11-10,00:46,BUSINESS,23F,420.0,2018-12-28,11:46:00,PAYPAL,COMPLETED,25.0,395.0,null,false
2254,4599869,FR2215,4296,738B738,91,92,22:15:00,2018-12-28,15:15:00,2018-12-26,02:13,BUSINESS,21B,524.0,2018-12-29,00:28:00,CREDIT CARD,COMPLETED,34.0,490.0,null,false
1203,9266529,UL0630,4349,320A320,102,100,06:30:00,2018-12-28,05:00:00,2018-11-01,00:56,BUSINESS,42E,432.0,2018-12-28,07:26:00,PAYPAL,CANCELLED,40.0,392.0,null,false
1914,5346479,EP1945,2923,100F100,86,85,19:45:00,2018-12-28,17:00:00,2018-12-10,02:06,BUSINESS,11D,516.0,2018-12-28,21:51:00,DEBIT CARD,CANCELLED,35.0,481.0,null,false
1473,2745249,AY1300,2350,320A320,99,80,13:00:00,2018-12-28,12:00:00,2018-12-26,03:01,BUSINESS,27A,582.0,2018-12-28,16:01:00,CASH,COMPLETED,69.84,512.16,null,false
